[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/ChrisW09/Python-for-AI-Driven-Automation/blob/main/03_real_world_io/12_apis_and_http.ipynb)

# 📓 Notebook 12 — APIs, HTTP, and Real-World Data Fetching

> **Module:** Data Engineering · **Estimated time:** 45–60 min · **Difficulty:** Beginner / Intermediate

So far the course has *talked* about API responses (NB 4). It's time to actually call a real API over the internet. This notebook is short on theory and long on practice: by the end you will have made dozens of HTTP requests, parsed the responses, handled errors, and built a small pipeline that pulls real weather data into a pandas DataFrame.

The patterns you learn here are the same ones you use to call **OpenAI, Anthropic, Stripe, Salesforce, your internal data warehouse, or any other modern service**. Once `requests.get(...)` feels routine, the entire web is reachable from your code.

## 🎯 Learning objectives

By the end of this notebook you can:

1. Send `GET` and `POST` requests with the **`requests`** library.
2. Read **HTTP status codes** and handle errors gracefully.
3. Add **query parameters**, **headers**, and **bearer-token auth**.
4. Parse **JSON** responses into Python dicts (and pandas DataFrames).
5. Handle **rate limits** and **flaky networks** with retry + backoff.
6. Paginate through a multi-page result set.
7. Build a small **ETL pipeline** that fetches data from an API and saves it locally.

## ✅ Prerequisites

Notebooks 1–11 (especially NB 2 for retry loops, NB 4 for JSON/dicts, and NB 7 for pandas — the ETL example builds a DataFrame).

> 💡 **No API keys required.** The two APIs we use — *Open-Meteo* (weather) and *JSONPlaceholder* (fake REST) — are free, public, and need no signup. If you have no internet access while running this notebook, every cell has a graceful fallback to a recorded response.

## 1. The shape of every HTTP request

```
                   ┌─── METHOD: GET | POST | PUT | DELETE
                   │   ┌── URL
                   │   │
                   │   │            ┌── HEADERS  (auth, content-type, …)
                   │   │            │
                   ▼   ▼            ▼
GET https://api.example.com/v1/weather   {"Authorization": "Bearer ..."}
                                          │
                          QUERY PARAMS    │  BODY  (POST/PUT only)
                          ?lat=52.5       │  {"name": "Alice"}
                          &lon=13.4       ▼
```

The four pieces above (method, URL, headers, body) describe **every** HTTP request you'll ever make — to a weather API, an LLM, a CRM, anything. The `requests` library wraps all of this in one function call per method: `requests.get(...)`, `requests.post(...)`, etc.

## 2. Setup

In [ ]:
import requests
import pandas as pd
from typing import Optional
import json
import time

print(f"requests version: {requests.__version__}")
print(f"pandas   version: {pd.__version__}")

# Two APIs we'll use throughout — both free, no key needed
WEATHER_API = "https://api.open-meteo.com/v1/forecast"
PLACEHOLDER  = "https://jsonplaceholder.typicode.com"

# A short timeout so a hanging network can't lock up our notebook
HTTP_TIMEOUT = 8   # seconds


## 3. Your first GET request

Let's ask the Open-Meteo API for the *current temperature in Berlin*.

In [ ]:
# 1. Build the parameters: latitude, longitude, what we want to know
params = {
    "latitude":  52.52,    # Berlin
    "longitude": 13.41,
    "current":   "temperature_2m",
}

# 2. Make the request
try:
    response = requests.get(WEATHER_API, params=params, timeout=HTTP_TIMEOUT)
except requests.exceptions.RequestException as e:
    print(f"⚠ Network unavailable ({type(e).__name__}). Using a recorded response.")
    payload = {
        "current": {"time": "2024-01-01T12:00", "temperature_2m": 4.3},
        "current_units": {"temperature_2m": "°C"},
    }
else:
    print(f"Status code: {response.status_code} ({response.reason})")
    print(f"URL called : {response.url}")
    payload = response.json()

# 3. Pull the field we care about
temp = payload["current"]["temperature_2m"]
unit = payload["current_units"]["temperature_2m"]
print(f"\nCurrent temperature in Berlin: {temp} {unit}")


**What just happened.**

1. `requests.get(url, params=...)` builds the URL `https://api.open-meteo.com/v1/forecast?latitude=52.52&longitude=13.41&current=temperature_2m` and sends a GET request.
2. The server returns a JSON document. `response.json()` parses it into a Python dict.
3. We drill into the dict with `[...]` (NB 4 patterns) to pull the value we want.

> 💡 **The `try / except` wrapping a real network call is non-negotiable.** Networks fail. Timeouts happen. APIs go down. Always have a fallback path — at minimum, fail loudly instead of hanging the kernel.

### 🔬 What actually happens — the request/response lifecycle

`requests.get(url)` looks like *one* magic step, but underneath it's just a polite, scripted **conversation**: you send one text message, the server sends one back, and the connection closes. Nothing more mysterious than that. Let's slow the whole exchange down to four moves.

```text
   YOUR CODE (the CLIENT)                              THE SERVER
   ─────────────────────                               ──────────

   requests.get(url, params=...)
            │
            │  ① FORMAT a request — a plain text message:
            │       GET /v1/forecast?lat=52.5 HTTP/1.1
            │       Host: api.open-meteo.com
            │       Accept: application/json
            │
            │  ───────────  request travels over the wire  ──────────▶  ② SERVER reads it,
            │                                                              looks up the data,
            │                                                              builds a reply
            │                                                                   │
            │  ◀──────────  response travels back  ─────────────────────────────┘
            │       HTTP/1.1 200 OK                       ③ REPLY — also plain text:
            │       Content-Type: application/json           status line + headers + body
            │
            │       {"temperature": 4.3}   ◀── the BODY (a JSON string)
            ▼
   ④ requests hands you a `response` object;
     response.json() turns the body STRING into a Python dict
```

The four pieces you send (**method · URL/path · headers · optional body**) and the three you get back (**status code · headers · body**) are the *entire* protocol. Every API call in this notebook — and every call to OpenAI, Stripe, or your warehouse — is exactly this shape.

### The request is just a dict; the response is just a dict

To make the lifecycle concrete (and to prove it needs no internet at all), we'll model it with plain Python. A **request** is four fields. A **response** is three. We'll write a tiny fake server — `handle(request) -> response` — that you could swap for the real `requests.get` without changing the idea.

| | A REQUEST has… | A RESPONSE has… |
|---|---|---|
| 1 | **method** — `GET`, `POST`, `PUT`, `DELETE` | **status** — a 3-digit code (e.g. `200`) |
| 2 | **url / path** — *what* you want | **headers** — metadata about the reply |
| 3 | **headers** — auth, content-type, who you are | **body** — the actual payload, as **text** |
| 4 | **body** — data you send (POST/PUT only) | *(that's it — 3 pieces)* |

> 🎯 **Intuition.** `requests.get(url)` only *formats the outgoing message and waits for the reply.* It doesn't "go get" anything itself — the **server** does the work and sends a message back. Your job is to format the request and decode the response.

In [ ]:
# ── OFFLINE proof: the full request/response lifecycle, no network ──
# We model the request as a dict, a fake server as handle(request)->response,
# and decode the reply exactly like real code does. Swap `handle` for
# `requests.get` and nothing about the *shape* changes.
import json


# ① FORMAT the request — method + path + headers + optional body
request = {
    "method":  "GET",
    "path":    "/v1/forecast?lat=52.52&lon=13.41",
    "headers": {"Accept": "application/json",
                "Authorization": "Bearer sk-demo-token"},
    "body":    None,                       # GET has no body
}

**Step 2 — the server: a tiny stand-in for the whole internet**

In [ ]:
# ② THE SERVER — a tiny stand-in for the whole internet.
#    Input: a request dict.  Output: a response dict (status + headers + body).
def handle(req):
    if req["headers"].get("Authorization") is None:
        return {"status": 401, "headers": {"Content-Type": "application/json"},
                "body": '{"error": "missing Authorization header"}'}
    if req["path"].startswith("/v1/forecast"):
        # NOTE: the body is a STRING — JSON is always text on the wire.
        return {"status": 200,
                "headers": {"Content-Type": "application/json"},
                "body": '{"temperature_2m": 4.3, "unit": "°C"}'}
    return {"status": 404, "headers": {"Content-Type": "application/json"},
            "body": '{"error": "not found"}'}

**Step 3 — send the request, then decode the text body**

In [ ]:
# ③ SEND it and get the reply back (this is the "requests.get(...)" moment)
response = handle(request)
print("status :", response["status"])
print("headers:", response["headers"])
print("body   :", response["body"], "  <-- this is a", type(response["body"]).__name__)

# ④ DECODE: the body is text — json.loads turns the STRING into a Python dict.
#    (This is literally what response.json() does for you under the hood.)
data = json.loads(response["body"])
print("\nparsed :", data, "  <-- now a", type(data).__name__)
print(f"It is {data['temperature_2m']}{data['unit']} in Berlin.")

### Reading the status code — the server's one-word verdict

Before you touch the body, you read the **status code**. It tells you, in one number, whether the conversation succeeded. The first digit sorts every code into one of four buckets:

| Class | Meaning | The ones you'll actually meet |
|-------|---------|-------------------------------|
| **2xx** | ✅ Success | `200 OK`, `201 Created`, `204 No Content` |
| **3xx** | ↪ Redirect — look elsewhere | `301 Moved Permanently`, `304 Not Modified` |
| **4xx** | 🙋 *You* messed up the request | `400 Bad Request`, `401 Unauthorized`, `403 Forbidden`, `404 Not Found`, `429 Too Many Requests` |
| **5xx** | 💥 The *server* messed up | `500 Internal Server Error`, `502 Bad Gateway`, `503 Service Unavailable` |

> 💡 **Tip — read it as a sentence.** `4xx` = *"your fault, fix the request"* (don't blindly retry — it'll fail the same way). `5xx` = *"their fault, maybe retry"*. `2xx` = *"all good, read the body"*. That single mental split drives all the retry logic later in this notebook.

The cell below decodes a status code purely from its first digit — no library, no network — to show the rule is just integer math.

In [ ]:
# ── OFFLINE proof: a status code's first digit IS its meaning ──
def classify(status: int) -> str:
    bucket = {2: "✅ success", 3: "↪ redirect",
              4: "🙋 client error (your fault)",
              5: "💥 server error (their fault)"}
    return bucket.get(status // 100, "❓ unknown")

NAMES = {200: "OK", 201: "Created", 301: "Moved Permanently",
         400: "Bad Request", 401: "Unauthorized", 404: "Not Found",
         429: "Too Many Requests", 500: "Internal Server Error",
         503: "Service Unavailable"}

for code_ in (200, 201, 301, 400, 401, 404, 429, 500, 503):
    print(f"{code_}  {NAMES[code_]:<22}  ->  {classify(code_)}")

# A response is only safe to parse on 2xx. Decide BEFORE reading the body:
def is_ok(status: int) -> bool:
    return 200 <= status < 300

print("\n200 ok? ", is_ok(200))     # True  -> go ahead and json.loads(body)
print("404 ok? ", is_ok(404))      # False -> handle the error instead


### Why headers matter — the envelope around the body

If the body is the *letter*, the **headers** are the *envelope*: small key–value lines that tell each side how to interpret what's inside. Two you'll use constantly:

- **`Content-Type`** — *what format the body is in.* `application/json` says "parse me with a JSON parser." If you ignore it and the server actually sent `text/html` (a styled error page), `json.loads` will choke — so checking `Content-Type` is how robust clients avoid a confusing `JSONDecodeError`.
- **`Authorization`** — *who you are.* The header `Authorization: Bearer <token>` is how OpenAI, Anthropic, GitHub, and Stripe know the request is really you. Leave it off a protected endpoint and you get `401 Unauthorized` — exactly what the fake server above is coded to return when the `Authorization` header is missing.

A header is *not* the data you asked for — it's **metadata about the data**: its format, its size, how to cache it, whether you're allowed to see it.

> 🧠 **Mental model.** *The internet is request-in, response-out.* You compose a text message (method + URL + **headers** + optional body), hand it off, and wait for a text message back (status + **headers** + body). `requests.get(url)` is just the courier that carries your envelope there and the reply back — the body only becomes a Python dict the instant **you** call `json.loads` (or `response.json()`) on it. Master those two envelopes and the entire web is just one function call away.

> ⚠️ **Pitfall.** The response body arrives as a **string**, never a ready-made dict. `data["temperature"]` on the raw `response.body` would index into characters of the *text*, not fields of an object. Always `json.loads(...)` (or `response.json()`) **first** — and only after you've confirmed a `2xx` status, since error responses (4xx/5xx) often have a body that isn't the JSON you expected.

## 4. HTTP status codes — what the server is telling you

You met the four status buckets in the deep dive above: **2xx** success, **3xx** redirect, **4xx** *your* mistake, **5xx** *their* mistake. Now let's see how to act on them in real code.

The single most important habit: **check the status code before using the response.** `requests` gives you two ways:

In [ ]:
# A) Check explicitly. Wrap the live call so the cell still runs offline.
try:
    r = requests.get(f"{PLACEHOLDER}/posts/1", timeout=HTTP_TIMEOUT)
    if r.status_code == 200:
        print(f"OK: got {len(r.text)} bytes")
    else:
        print(f"Error: {r.status_code}")
except requests.exceptions.RequestException:
    print("Offline — skipping live check")

# B) Use raise_for_status() — raises HTTPError for 4xx / 5xx
try:
    r = requests.get(f"{PLACEHOLDER}/this-does-not-exist", timeout=HTTP_TIMEOUT)
    r.raise_for_status()
except requests.HTTPError as e:
    print(f"\nHTTPError caught: {e}")
except requests.exceptions.RequestException as e:
    print(f"\nOther network error: {type(e).__name__}: {e}")


> 🎯 **Rule of thumb.** Use `r.raise_for_status()` inside a `try / except requests.HTTPError` block. It keeps your business logic clean — the moment any 4xx / 5xx arrives, control jumps to the error handler.

## 5. Headers, authentication, and `User-Agent`

Most production APIs require some form of authentication via a request **header**. The two patterns you'll see 95% of the time:

```python
# Bearer token (OpenAI, Anthropic, GitHub, most modern APIs)
headers = {"Authorization": f"Bearer {API_KEY}"}

# API key in a custom header (older APIs)
headers = {"X-API-Key": API_KEY}

requests.get(url, headers=headers, timeout=8)
```

Free public APIs like Open-Meteo don't require auth, but they often *do* appreciate a user-agent so they know who's calling. Let's set one.

In [ ]:
# Setting custom headers — same pattern you'll use for bearer-token auth
import os

headers = {
    # Identify yourself politely
    "User-Agent": "python-for-ai-course/1.0 (learning HTTP requests)",
    # When you eventually need a real API:
    # "Authorization": f"Bearer {os.getenv('OPENAI_API_KEY', 'sk-...')}"
}

try:
    r = requests.get(f"{PLACEHOLDER}/users/1", headers=headers, timeout=HTTP_TIMEOUT)
    r.raise_for_status()
    user = r.json()
    print(f"User #{user['id']}: {user['name']}")
    print(f"  email: {user['email']}")
    print(f"  city : {user['address']['city']}")
except requests.exceptions.RequestException:
    print("⚠ Network unavailable. Recorded value: 'Leanne Graham'")


> ⚠️ **Never put API keys in your notebook or git history.** Read them from environment variables (`os.getenv('OPENAI_API_KEY')`) or a `.env` file with `python-dotenv`. The comment line above shows the pattern.

## 6. POST — sending data to a server

`POST` is for *creating* something. The data goes in the request **body**, usually as JSON.

In [ ]:
# Create a new post on the fake-API server
new_post = {
    "title":  "What I learned about HTTP today",
    "body":   "Status codes, headers, retries, pagination. All of it.",
    "userId": 1,
}

try:
    r = requests.post(f"{PLACEHOLDER}/posts", json=new_post, timeout=HTTP_TIMEOUT)
    r.raise_for_status()
    created = r.json()
    print(f"Created post #{created.get('id')} (status {r.status_code} {r.reason})")
    print(f"  title: {created.get('title')}")
except requests.exceptions.RequestException:
    print("⚠ Network unavailable. (POST would have returned id=101)")


> 💡 `json=new_post` tells `requests` to JSON-encode your dict and set the right `Content-Type` header. The alternative `data=...` parameter sends form-encoded data instead.

## 7. Retry with exponential backoff

Networks are *unreliable*. A polite client retries with increasing delays — usually `0.5s → 1s → 2s → 4s → ...`. This is exactly the pattern from NB 2, dressed up for HTTP.

In [ ]:
def fetch_with_retry(url: str,
                     params: Optional[dict] = None,
                     max_attempts: int = 4,
                     base_delay: float = 0.5) -> Optional[dict]:
    """GET a URL with exponential backoff. Returns the JSON payload or None."""
    for attempt in range(1, max_attempts + 1):
        try:
            r = requests.get(url, params=params, timeout=HTTP_TIMEOUT)
            # 429 = rate-limited, 5xx = server problem — both worth retrying
            if r.status_code == 429 or 500 <= r.status_code < 600:
                raise requests.HTTPError(f"server said {r.status_code}", response=r)
            r.raise_for_status()
            return r.json()
        except requests.exceptions.RequestException as e:   # includes HTTPError
            if attempt == max_attempts:
                print(f"  attempt {attempt}: giving up ({e})")
                return None
            wait = base_delay * 2 ** (attempt - 1)
            print(f"  attempt {attempt}: {e}  — waiting {wait:.2f}s")
            time.sleep(wait)
    return None


# Try a few endpoints — the second one is intentionally broken
for url in [f"{PLACEHOLDER}/users/1", f"{PLACEHOLDER}/this-route-does-not-exist"]:
    print(f"\nGET {url}")
    data = fetch_with_retry(url, max_attempts=2)
    print(f"  result: {type(data).__name__}{' (ok)' if data else ' (failed)'}")


> 🎯 **Which errors do you retry?**
> - Retry: **network timeouts**, **429 Too Many Requests**, **5xx** server errors.
> - **Don't** retry: **400 Bad Request**, **401 Unauthorized**, **404 Not Found** — these will fail again on the next try. Fix the request instead.

## 8. Pagination — when one request isn't enough

Most APIs return at most ~100 items per request. To get more, you **paginate** — making one request per page until you've seen everything.

Two common pagination schemes:

| Scheme | How it works | API examples |
|---|---|---|
| **Page number** | `?page=1&per_page=50`, then `?page=2`, … | GitHub, many REST APIs |
| **Cursor**       | response has a `next_cursor`; pass it in the next call | Twitter/X, Slack, modern APIs |

We'll demo the page-number pattern.

In [ ]:
def fetch_all_pages(base_url: str, per_page: int = 10, max_pages: int = 20) -> list:
    """Fetch every page from a page-number-style endpoint."""
    all_items = []
    for page in range(1, max_pages + 1):
        try:
            r = requests.get(base_url,
                             params={"_page": page, "_limit": per_page},
                             timeout=HTTP_TIMEOUT)
            r.raise_for_status()
            items = r.json()
        except requests.exceptions.RequestException:
            print(f"  page {page}: network error, stopping")
            break
        if not items:                # empty page → we're done
            print(f"  page {page}: empty, stopping")
            break
        all_items.extend(items)
        print(f"  page {page}: fetched {len(items)} items (total {len(all_items)})")
    return all_items


# JSONPlaceholder has 100 posts; we'll grab them 25 at a time
posts = fetch_all_pages(f"{PLACEHOLDER}/posts", per_page=25)
print(f"\n→ Total posts fetched: {len(posts)}")


> 💡 **Always cap your loop.** `max_pages=20` is a safety net. Without it, a buggy API that always returns the same page would loop forever.

## 9. Putting it together — a real ETL pipeline

Let's build a tiny **E**xtract-**T**ransform-**L**oad job:

1. **Extract**: fetch a 7-day weather forecast for several cities.
2. **Transform**: flatten the response into one tidy row per (city, day).
3. **Load**: save to a CSV that any later notebook can read.

In [ ]:
CITIES = [
    ("Berlin",    52.52, 13.41),
    ("London",    51.51, -0.13),
    ("New York",  40.71, -74.01),
    ("Tokyo",     35.68, 139.65),
]

def fetch_forecast(city: str, lat: float, lon: float) -> Optional[pd.DataFrame]:
    """Get a 7-day daily forecast for one city, return as a DataFrame."""
    params = {
        "latitude": lat, "longitude": lon,
        "daily": "temperature_2m_max,temperature_2m_min,precipitation_sum",
        "timezone": "auto",
        "forecast_days": 7,
    }
    data = fetch_with_retry(WEATHER_API, params=params, max_attempts=3)
    if data is None or "daily" not in data:
        return None
    df = pd.DataFrame(data["daily"])
    df["city"] = city
    return df

**Run the pipeline — loop over cities, then combine the frames**

In [ ]:
# Run the pipeline
frames = []
for city, lat, lon in CITIES:
    print(f"→ Fetching forecast for {city} ...")
    f = fetch_forecast(city, lat, lon)
    if f is not None:
        frames.append(f)

if frames:
    forecast = pd.concat(frames, ignore_index=True)
    forecast = forecast[["city", "time", "temperature_2m_max",
                          "temperature_2m_min", "precipitation_sum"]]
    print(f"\n✅ Combined DataFrame: {forecast.shape}")
    print(forecast.head(10))
else:
    print("⚠ No data fetched (offline?) — try again with internet.")

In [ ]:
# Save the result so other notebooks can use it
if frames:
    out_path = "forecast.csv"
    forecast.to_csv(out_path, index=False)
    print(f"Wrote {out_path}  ({len(forecast)} rows)")


**Read that pipeline carefully.** It is short, but it is *real*:

- One function per responsibility (`fetch_forecast` extracts + does an initial transform).
- The retry / fallback logic from earlier is reused without modification.
- The output is a clean CSV that integrates with everything else you've learned (NB 7).

This is the canonical shape of every data-pull script you will ever write. The only thing that changes between projects is the URL and the schema.

## 10. Common pitfalls

| Pitfall | Symptom | Fix |
|---|---|---|
| No timeout on the request | notebook hangs forever | Always pass `timeout=...` |
| API key in the code | leaks to git history | Use `os.getenv(...)` |
| Hammering an API in a tight loop | get rate-limited (`429`) | Add `time.sleep(...)` between calls |
| Parsing JSON without checking status | `JSONDecodeError` on error pages | Call `r.raise_for_status()` first |
| One bad row kills the whole batch | exception crashes the loop | Wrap each call in `try / except` |
| Retrying a 401 / 404 | wastes time, won't change | Retry only for `429` and `5xx` |

## 🧪 Practice exercises

### Exercise 1 — ⭐ A simple `safe_fetch`

Write `safe_fetch(url)` that:

1. Sends a GET with `timeout=8`.
2. Returns the parsed JSON on `200`.
3. Returns `None` (no crash) on any error.
4. Prints a one-line warning so a human knows what happened.

Test it on `f"{PLACEHOLDER}/users/2"` (works) and `f"{PLACEHOLDER}/no-such-thing"` (404).

In [ ]:
# Your code here  👇
def safe_fetch(url):
    pass


<details>
<summary>💡 <b>Solution</b></summary>

```python
def safe_fetch(url: str):
    try:
        r = requests.get(url, timeout=HTTP_TIMEOUT)
        r.raise_for_status()
        return r.json()
    except requests.HTTPError as e:
        print(f"⚠ HTTP {e.response.status_code} for {url}")
    except requests.exceptions.RequestException as e:
        print(f"⚠ Network error: {type(e).__name__}")
    return None

print(safe_fetch(f"{PLACEHOLDER}/users/2"))     # works
print(safe_fetch(f"{PLACEHOLDER}/no-such-thing"))  # 404 → None, no crash
```

**Reasoning.** A defensive wrapper at the boundary of your code (where it meets the outside world) keeps the rest of your script simple. Errors are visible (the warning print) but non-fatal.
</details>

### Exercise 2 — ⭐⭐ Build a tiny user report

Use the JSONPlaceholder API (`{PLACEHOLDER}/users`) to:

1. Fetch all users (no pagination needed — the endpoint returns 10).
2. Build a DataFrame with columns `id`, `name`, `email`, `city`, `company`.
3. Print the unique cities they live in.

In [ ]:
# Your code here  👇


<details>
<summary>💡 <b>Solution</b></summary>

```python
data = safe_fetch(f"{PLACEHOLDER}/users") or []

rows = [{
    "id":      u["id"],
    "name":    u["name"],
    "email":   u["email"],
    "city":    u["address"]["city"],
    "company": u["company"]["name"],
} for u in data]

df_users = pd.DataFrame(rows)
print(df_users)
print(f"\nUnique cities: {df_users['city'].unique().tolist()}")
```

**Pattern used.** Flatten nested JSON into a flat row dict with a list comprehension. We'll see this exact idiom whenever an API returns deeply-nested objects — it converts JSON-land into table-land in one expression.
</details>

### Exercise 3 — ⭐⭐ Rate-limited polite client

Wrap `safe_fetch` in `polite_fetch_all(urls, delay=0.2)` that calls each URL with a small delay between calls — so you never hammer the server. Then fetch posts 1–5 from JSONPlaceholder.

In [ ]:
# Your code here  👇


<details>
<summary>💡 <b>Solution</b></summary>

```python
def polite_fetch_all(urls, delay=0.2):
    results = []
    for i, url in enumerate(urls):
        if i > 0:
            time.sleep(delay)
        results.append(safe_fetch(url))
    return results

urls = [f"{PLACEHOLDER}/posts/{i}" for i in range(1, 6)]
posts = polite_fetch_all(urls, delay=0.1)
for p in posts:
    if p:
        print(f"#{p['id']}: {p['title'][:50]}…")
```

**Why this matters.** Every API has a rate limit — some explicit (Twitter, Slack), some implicit (Open-Meteo will silently throttle you above a few requests per second). A small `time.sleep` between calls is the difference between *useful client* and *blocked client*.
</details>

### Exercise 4 — ⭐⭐ Debug me 🐞

The function below is supposed to return the title of a JSONPlaceholder post, but it sometimes crashes mysteriously. Find the bug(s).

In [ ]:
# 👇 Your fixed/corrected version goes here — write or paste it below.
def get_title(post_id):
    r = requests.get(f"{PLACEHOLDER}/posts/{post_id}")
    return r.json()["title"]

# This call works...
print(get_title(1))
# ...but this one fails with a confusing error if the server is slow or the id is bad.
# print(get_title(99999))


<details>
<summary>💡 <b>Solution</b></summary>

Three problems:

1. **No timeout.** A slow server would hang the notebook indefinitely.
2. **No status-code check.** `requests.get(.../posts/99999)` returns `404` with an empty `{}` body — `r.json()["title"]` then raises `KeyError`, not a clear HTTP error.
3. **No error handling.** A network blip propagates as an obscure traceback.

```python
def get_title(post_id):
    try:
        r = requests.get(f"{PLACEHOLDER}/posts/{post_id}", timeout=HTTP_TIMEOUT)
        r.raise_for_status()
        return r.json().get("title")     # .get() returns None if missing
    except requests.exceptions.RequestException as e:
        print(f"⚠ Failed to fetch post {post_id}: {e}")
        return None
```

**Lesson.** Every external call has three failure modes — *slow*, *unavailable*, *unexpected response*. A 4-line function should defend against all three.
</details>

## 🧠 Stretch exercises

Two more applied exercises to deepen the material. Try them yourself before opening the solution.


### Stretch exercise A — ⭐⭐⭐ A bearer-token client (mocked)

Write `make_client(api_key)` that returns a `requests.Session` pre-configured with a bearer-token `Authorization` header and a default 8-second timeout. Use it to make one mock GET request and show that the header was sent.


In [ ]:
# Your code here  👇
import requests
API_KEY = "sk-fake-1234"


<details>
<summary>💡 <b>Solution</b></summary>

```python
import requests

def make_client(api_key: str, timeout: float = 8.0) -> requests.Session:
    sess = requests.Session()
    sess.headers.update({
        "Authorization": f"Bearer {api_key}",
        "User-Agent":    "python-for-ai-course/1.0",
    })
    # `Session` doesn't accept a default timeout — wrap requests instead
    sess.request = (lambda old:
                     lambda method, url, **kw: old(method, url, timeout=kw.pop("timeout", timeout), **kw))(sess.request)
    return sess


client = make_client(API_KEY)

# Make a real call to JSONPlaceholder and confirm the Authorization header is set
try:
    r = client.get(f"{PLACEHOLDER}/users/1")
    print("Sent Authorization header:", r.request.headers.get("Authorization"))
    print("Status:", r.status_code)
except requests.exceptions.RequestException:
    print("(offline) — session.headers would have included the Authorization line.")
```

**Two production habits in one snippet.**
1. **Sessions** keep TCP connections alive across requests — much
   faster than `requests.get` in a loop.
2. **Shared headers** mean every call carries auth + a user-agent
   without you remembering. The default-timeout shim above is a
   common pattern when wrapping an HTTP client.

</details>

### Stretch exercise B — ⭐⭐⭐ Polite rate limiter

Write `rate_limited(min_interval_s)`: a decorator that ensures the wrapped function is called at most once every `min_interval_s` seconds, blocking with `time.sleep` if necessary. Test it on a function that prints a timestamp.


In [ ]:
# Your code here  👇
import time
import functools


<details>
<summary>💡 <b>Solution</b></summary>

```python
import time, functools

def rate_limited(min_interval_s: float):
    last_called = [0.0]
    def deco(fn):
        @functools.wraps(fn)
        def wrapper(*args, **kwargs):
            wait = min_interval_s - (time.time() - last_called[0])
            if wait > 0:
                time.sleep(wait)
            last_called[0] = time.time()
            return fn(*args, **kwargs)
        return wrapper
    return deco


@rate_limited(0.2)
def stamp(i):
    print(f"  call {i} @ {time.strftime('%H:%M:%S')}.{int((time.time()%1)*1000):03d}")

t0 = time.time()
for i in range(5):
    stamp(i)
print(f"\nTotal elapsed: {time.time() - t0:.2f}s (expected ≥ 0.8s)")
```

**Why this matters.** Free APIs often *silently* throttle if you
call too fast — your error is a 429 *the next time*, not the first.
A 5-line rate limiter is the cheapest insurance.

</details>

### Stretch exercise C — ⭐⭐⭐ Defensive JSON parser

API responses lie. Write a function `safe_get(d, *keys, default=None)` that walks a nested dict by a sequence of keys and returns `default` if any key is missing or any value along the way isn't a dict.

```python
data = {"user": {"profile": {"name": "Ada"}}}
safe_get(data, "user", "profile", "name")     # → 'Ada'
safe_get(data, "user", "settings", "theme")    # → None
safe_get(data, "user", "profile", "name", "first")  # → None  (name is a string, not a dict)
```

In [ ]:
# Your code here  👇
def safe_get(d, *keys, default=None):
    ...

data = {"user": {"profile": {"name": "Ada"}}}
print(safe_get(data, "user", "profile", "name"))
print(safe_get(data, "user", "settings", "theme"))


<details>
<summary>💡 <b>Solution — click to expand</b></summary>

```python
def safe_get(d, *keys, default=None):
    for k in keys:
        if not isinstance(d, dict) or k not in d:
            return default
        d = d[k]
    return d
```

**Reasoning.** Three details worth absorbing. (1) The `isinstance(d, dict)` guard catches the case where you descend into a non-dict value (a string, a list, `None`) — without it you'd raise `TypeError`. (2) Reassigning `d = d[k]` walks the structure one key at a time — concise and intuitive. (3) Returning a single `default` sentinel beats raising in API-parsing code: callers can use `or` (`name = safe_get(d, ...) or "anonymous"`) or compare against the sentinel as they prefer. The standard library's `dict.get` only handles one level; this is the multi-level version you'll write again and again.
</details>

### Stretch exercise D — ⭐⭐⭐ Paginate through an API

Many APIs return data in pages with a `next_url` field, e.g.:

```python
page_1 = {"results": [1, 2, 3],   "next_url": "https://api.example.com/items?page=2"}
page_2 = {"results": [4, 5, 6],   "next_url": "https://api.example.com/items?page=3"}
page_3 = {"results": [7, 8],      "next_url": None}
```

Write `collect_all_pages(fetch, start_url)` where `fetch(url)` returns a page dict. It should return the **flat** list of all results across pages. To keep this offline, use the stub `fetch` in the skeleton.

In [ ]:
# Your code here  👇
PAGES = {
    "https://api/p1": {"results": [1, 2, 3],  "next_url": "https://api/p2"},
    "https://api/p2": {"results": [4, 5, 6],  "next_url": "https://api/p3"},
    "https://api/p3": {"results": [7, 8],     "next_url": None},
}
def fetch(url): return PAGES[url]

def collect_all_pages(fetch, start_url):
    ...

# print(collect_all_pages(fetch, "https://api/p1"))


<details>
<summary>💡 <b>Solution — click to expand</b></summary>

```python
def collect_all_pages(fetch, start_url):
    items = []
    url = start_url
    while url:
        page = fetch(url)
        items.extend(page["results"])
        url = page.get("next_url")
    return items

print(collect_all_pages(fetch, "https://api/p1"))  # [1,2,3,4,5,6,7,8]
```

**Reasoning.** Pagination is one of those patterns you'll write more times than you'd think. Two design choices to internalise. (1) Take `fetch` as a parameter rather than calling `requests.get` directly — this makes the function trivially testable with a fake fetcher (which is exactly what we did here). (2) Loop on `url` rather than counting pages — the API tells you when to stop by returning `next_url = None`. Don't invent your own stop condition (`page < 100`) unless you also log a warning when it triggers; runaway loops in production rack up real API bills.
</details>

## 🎁 Bonus mini-project — A weekly weather dashboard

Build a function `weather_dashboard(cities)` that:

1. Takes a list of `(name, lat, lon)` tuples.
2. Fetches the 7-day forecast for each.
3. Returns one tidy DataFrame with columns `city`, `date`, `tmax`, `tmin`, `precip_mm`.
4. Prints a small summary: hottest city next week, wettest city next week.

You already have most of the building blocks — `fetch_forecast` from §9 does the per-city work.

In [ ]:
# Your code here  👇


<details>
<summary>💡 <b>Solution</b></summary>

```python
def weather_dashboard(cities):
    frames = []
    for city, lat, lon in cities:
        df = fetch_forecast(city, lat, lon)
        if df is not None:
            df = df.rename(columns={"time": "date",
                                     "temperature_2m_max": "tmax",
                                     "temperature_2m_min": "tmin",
                                     "precipitation_sum":  "precip_mm"})
            frames.append(df[["city", "date", "tmax", "tmin", "precip_mm"]])
    if not frames:
        print("(no data — offline?)")
        return pd.DataFrame()

    all_data = pd.concat(frames, ignore_index=True)
    hottest = all_data.loc[all_data["tmax"].idxmax()]
    wettest = all_data.groupby("city")["precip_mm"].sum().idxmax()
    print(f"☀ Hottest day next week : {hottest['city']} on {hottest['date']} ({hottest['tmax']}°C)")
    print(f"🌧 Wettest city next week: {wettest}")
    return all_data

dash = weather_dashboard(CITIES)
dash.head()
```

**What you just built.** A complete, real-data analytical pipeline: extract from a live API, transform into a tidy table, summarise for a human. Add one more line — `dash.pivot(index="date", columns="city", values="tmax").plot()` — and you have your first daily weather chart (NB 9 style: one line, no boilerplate).
</details>

## 🧠 Key takeaways

1. Every HTTP request is **method + URL + headers + (optional body)**. The `requests` library wraps it in one function call per method.
2. **Always pass `timeout=...`** — without it, your code can hang forever.
3. **Always check the status code** with `r.raise_for_status()` before using the response.
4. Retry **429** and **5xx**; never retry **4xx** (other than 429) — fix the request instead.
5. Use **environment variables** for API keys, never inline them.
6. **`json=...`** for POST bodies, **`params=...`** for query strings, **`headers=...`** for auth.
7. The shape of every real-world data-fetch script: **extract → transform → load** (ETL).
8. Defensive wrappers at the **boundary** of your code keep the **inside** simple.

## ✅ Self-assessment

- [ ] Make a GET request with query parameters and headers
- [ ] Read and act on HTTP status codes
- [ ] Use `raise_for_status()` inside a `try / except`
- [ ] Send a POST request with a JSON body
- [ ] Write a retry loop with exponential backoff for transient errors
- [ ] Paginate through a multi-page result set
- [ ] Flatten a nested JSON response into a flat DataFrame

## 🚀 Next step

Continue with **Notebook 13 — SQL Fundamentals with pandas**, where the CSV you just produced becomes the input to a small database, and you'll see when SQL is faster (and clearer) than pandas — and when it isn't.